# Final model selection

This notebook selects the feature set and the model that the app will use. All candidates go through one protocol.

- **Our best model from week 6:** random forest + school districts + comparable sales (`experiments/05e_comparable_sales.ipynb`).
- **The team unified pipeline:** XGBoost + school districts + HOA + flooring (`docs/team/`). The team reported a validation R² of 0.909, but the team used a different split and different cleaning. Thus, we cannot compare that value with our values.
- **Not tested before:** HOA + flooring **and** comparable sales together.

**Protocol** (the same as all notebooks since week 5):
- We use the 6 validation months 2025-10 to 2026-03. For each month, the model trains on the 18 months before it.
- We do not score the test month (2026-04) here. We score it one time, for the selected model, after this notebook.
- Dataset A follows the team's unified cleaning (`clean_source` in `docs/team/`), plus two extra removals: rows without a valid address and duplicate sales by address and close date. Each `ListingKey` occurs one time, so a sale cannot be in training and again in validation.
- All features are known before listing. The price trim, the imputation and the target encoding are fit on the training window only.

**Decision metric: median absolute percentage error (MdAPE).** The goal is an accurate estimate for a typical home. R² and the dollar error give the most weight to expensive homes. MdAPE gives the same weight to each home, and a few very large errors do not change it. We also report R².

**Decision rule:** a candidate is better than the reference if its mean MdAPE is lower **and** its MdAPE is lower in at least 5 of the 6 months.

**Feature sets.** All feature sets include Dataset A. The school districts come in two versions:
- *School districts:* our spatial join of the home coordinates to the California district boundaries.
- *MLS district:* the high school district from the listing text. The team model used this version. In our data, the MLS elementary district column is empty.

Both versions also include the share of disadvantaged students in the district (`DistrictSEDPct`), as in the team model.

**Models.**
- Random forest: the forest that we use since week 5.
- XGBoost (06 settings): depth 8, 600 trees, learning rate 0.092, column sampling 0.5. We tuned these settings in `06` on 2025-09. The team document gives almost the same settings (learning rate 0.1).
- XGBoost (team settings): depth 7, 600 trees, learning rate 0.1, no column sampling, seed 42. These settings come from the team's saved model (`data/models/teammate_xgboost_school_hoa_flooring.joblib`). They are not the same as the settings in the team document.

**Target.** Each candidate is fit two times: on the dollar price and on the log price. A log target gives the same weight to a 10% error at all prices, so it can decrease the percentage error.

In [1]:
# Import the boosting library before anything that loads torch (OpenMP conflict on macOS).
from xgboost import XGBRegressor

import time

import numpy as np
import pandas as pd

from idx_exchange_data_science import features as ft
from idx_exchange_data_science.cleaning import DATE_COL, TARGET
from idx_exchange_data_science.features import build_table
from idx_exchange_data_science.modeling import backtest, build_model, gain_summary, random_forest, score, tuned_xgboost
from idx_exchange_data_science.paths import ARCHIVE_DIR, DATA_DIR, RESULTS_DIR
from idx_exchange_data_science.preprocessing import make_preprocessor
from idx_exchange_data_science.splits import validation_months

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_colwidth", 80)

X = 18
EVAL_MONTHS = validation_months("2026-04", n_months=6)  # 2025-10 to 2026-03; 2026-04 is the held-out test month
DECISION_METRIC = "median_ape"
METRICS = ["median_ape", "r2", "mae", "mape", "within_10pct"]
TRAIN_METRICS = ["train_median_ape", "train_r2"]  # in-sample, to show overfitting

df = build_table()
print("validation months:", [str(m) for m in EVAL_MONTHS], "| rows:", len(df))

validation months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03'] | rows: 308786


## 1. Candidates

In [2]:
HOA_FLOORING = ft.HOA_COLS + ft.FLOORING_COLS
FEATURE_SETS = {  # name: (numeric columns, categorical columns), in addition to Dataset A
    "Districts + comps": (ft.DISTRICT_NUMERIC_COLS + ft.COMP_COLS, ft.DISTRICT_COLS),
    "Districts + HOA + flooring": (ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING, ft.DISTRICT_COLS),
    "Districts + HOA + flooring + comps": (ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING + ft.COMP_COLS, ft.DISTRICT_COLS),
    "MLS district + HOA + flooring": (ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING, ft.MLS_DISTRICT_COLS),
    "MLS district + HOA + flooring + comps": (ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING + ft.COMP_COLS, ft.MLS_DISTRICT_COLS),
}
GRID = list(FEATURE_SETS)  # every model and target is tried on these
# Added later, for the best model and target only (section 6).
FEATURE_SETS |= {
    "Districts + HOA + flooring + similar comps": (ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING + ft.SIMILAR_COLS, ft.DISTRICT_COLS),
    "Districts + HOA + flooring + comps + similar comps":
        (ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING + ft.COMP_COLS + ft.SIMILAR_COLS, ft.DISTRICT_COLS),
    "Districts + HOA + flooring + comps + comp median price":
        (ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING + ft.COMP_COLS + ft.COMP_PRICE_COLS, ft.DISTRICT_COLS),
}
SIMILAR_SETS = [name for name in FEATURE_SETS if name not in GRID]  # section 6
# Section 7: the team's default room ratios, added to the section 6 winner.
FEATURE_SETS["Districts + HOA + flooring + comps + comp median price + ratios"] = (
    ft.DISTRICT_NUMERIC_COLS + HOA_FLOORING + ft.COMP_COLS + ft.COMP_PRICE_COLS + ft.RATIO_COLS, ft.DISTRICT_COLS)
ESTIMATORS = {
    "Random Forest": random_forest,
    "XGBoost (06 settings)": tuned_xgboost,
    "XGBoost (team settings)": lambda: tuned_xgboost(learning_rate=0.1, max_depth=7, colsample_bytree=1.0, random_state=42),
}
TARGETS = {"dollar price": False, "log price": True}
REFERENCE = "Random Forest | Districts + comps | dollar price"


def candidate(estimator, feature_set, log_target):
    numeric, categorical = FEATURE_SETS[feature_set]
    return build_model(ESTIMATORS[estimator](), make_preprocessor(numeric, categorical), log_target=log_target)


pd.DataFrame([{"feature set": name, "numeric (in addition to Dataset A)": ", ".join(num), "categorical": ", ".join(cat)}
              for name, (num, cat) in FEATURE_SETS.items()])

,feature set,numeric (in addition to Dataset A),categorical
0,Districts + comps,"DistrictSEDPct, LogCompPricePerSqft, LogCompImpliedPrice, CompDistanceKm","ElementaryDistrict, HighDistrict"
1,Districts + HOA + flooring,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...","ElementaryDistrict, HighDistrict"
2,Districts + HOA + flooring + comps,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...","ElementaryDistrict, HighDistrict"
3,MLS district + HOA + flooring,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...",MLSHighSchoolDistrict
4,MLS district + HOA + flooring + comps,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...",MLSHighSchoolDistrict
5,Districts + HOA + flooring + similar comps,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...","ElementaryDistrict, HighDistrict"
6,Districts + HOA + flooring + comps + similar comps,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...","ElementaryDistrict, HighDistrict"
7,Districts + HOA + flooring + comps + comp median price,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...","ElementaryDistrict, HighDistrict"
8,Districts + HOA + flooring + comps + comp median price + ratios,"DistrictSEDPct, HOAFeeMonthly, HasHOA, HOAFeeMissing, FlooringWood, Flooring...","ElementaryDistrict, HighDistrict"


## 2. Validation backtest
34 candidates x 6 months. Most of the run time is for the random forests.

**Saved results.** The monthly scores and the predictions of each candidate are saved. A new run fits only the candidates that are not saved yet. Thus, to add an experiment, add it to the lists above and run the notebook again. Each candidate name describes its full configuration, so a saved result is used only for the same configuration. After a change to the data, the features or the evaluation setup, set `REFIT_ALL = True` (or delete the saved files) to fit all candidates again.

In [3]:
SCORES_PATH = ARCHIVE_DIR / "final_model_selection_backtest.csv"
PREDICTIONS_PATH = DATA_DIR / "experiments" / "final_model_selection_predictions.parquet"  # gitignored, ~15 MB
REFIT_ALL = False

CANDIDATES = {f"{e} | {f} | {t}": (e, f, log) for e in ESTIMATORS for f in GRID for t, log in TARGETS.items()}
CANDIDATES |= {f"XGBoost (06 settings) | {f} | log price": ("XGBoost (06 settings)", f, True) for f in FEATURE_SETS if f not in GRID}
saved = not REFIT_ALL and SCORES_PATH.exists() and PREDICTIONS_PATH.exists()
scores = pd.read_csv(SCORES_PATH) if saved else pd.DataFrame(columns=["model"])
predictions = pd.read_parquet(PREDICTIONS_PATH) if saved else pd.DataFrame()
todo = [name for name in CANDIDATES if name not in set(scores["model"]) or name not in predictions.columns]
print(f"{len(CANDIDATES) - len(todo)} candidates loaded from saved results, {len(todo)} to fit")

for name in todo:
    start = time.perf_counter()
    month_scores, month_predictions = backtest({name: candidate(*CANDIDATES[name])}, df, EVAL_MONTHS, X)
    month_scores["fit_seconds"] = (time.perf_counter() - start) / len(EVAL_MONTHS)
    scores = pd.concat([scores[scores["model"] != name], month_scores], ignore_index=True)
    predictions = predictions.drop(columns=name, errors="ignore").join(month_predictions, how="outer")
    scores.to_csv(SCORES_PATH, index=False)  # saved after each candidate, so a stopped run loses little
    PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    predictions.to_parquet(PREDICTIONS_PATH)

scores = scores[scores["model"].isin(CANDIDATES)].reset_index(drop=True)
predictions = predictions[list(CANDIDATES)]
scores[["estimator", "feature_set", "target"]] = scores["model"].str.split(" | ", regex=False, expand=True)
actual = df.loc[predictions.index, TARGET]

0 candidates loaded from saved results, 34 to fit


In [4]:
summary = scores.groupby("model")[METRICS + TRAIN_METRICS].mean()
summary = summary.join(gain_summary(scores, reference=REFERENCE, metric=DECISION_METRIC)).sort_values(DECISION_METRIC)
summary["fit seconds / month"] = scores.groupby("model")["fit_seconds"].mean()
summary

,median_ape,r2,mae,mape,within_10pct,train_median_ape,train_r2,mean gain,std gain,months improved (of 6),fit seconds / month
model,,,,,,,,,,,
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,0.0702,0.9144,"133,820.4476",0.1044,0.6358,0.0573,0.9620,-0.0084,0.0011,6.0000,5.1686
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,0.0707,0.9145,"134,086.2215",0.1047,0.6340,0.0568,0.9633,-0.0078,0.0009,6.0000,5.4989
XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,0.0711,0.9142,"135,091.0967",0.1055,0.6317,0.0571,0.9625,-0.0074,0.0013,6.0000,5.5983
XGBoost (06 settings) | Districts + HOA + flooring + similar comps | log price,0.0715,0.9109,"136,353.2836",0.1061,0.6316,0.0581,0.9601,-0.0070,0.0015,6.0000,5.0693
XGBoost (06 settings) | MLS district + HOA + flooring + comps | log price,0.0720,0.9123,"136,370.3106",0.1062,0.6291,0.0586,0.9598,-0.0065,0.0018,6.0000,4.8086
XGBoost (06 settings) | Districts + HOA + flooring + comps | log price,0.0721,0.9133,"135,999.7839",0.1063,0.6278,0.0587,0.9600,-0.0064,0.0017,6.0000,4.8940
XGBoost (team settings) | Districts + HOA + flooring + comps | log price,0.0731,0.9129,"137,026.0104",0.1072,0.6235,0.0626,0.9523,-0.0054,0.0014,6.0000,4.4471
XGBoost (team settings) | MLS district + HOA + flooring + comps | log price,0.0732,0.9134,"137,361.8071",0.1075,0.6229,0.0626,0.9525,-0.0053,0.0014,6.0000,4.4178
XGBoost (team settings) | MLS district + HOA + flooring | log price,0.0735,0.9057,"140,470.6788",0.1087,0.6207,0.0643,0.9467,-0.0051,0.0018,6.0000,4.0030


`train_median_ape` and `train_r2` are the scores on each model's own training rows. A large difference from the validation scores shows overfitting.

Mean MdAPE for each feature set, model and target (lower is better):

In [5]:
table = scores.pivot_table(index="feature_set", columns=["estimator", "target"], values=DECISION_METRIC, aggfunc="mean")
table = table.reindex(index=list(FEATURE_SETS), columns=[(e, t) for e in ESTIMATORS for t in TARGETS])
table.style.format("{:.2%}").background_gradient(cmap="Blues_r", axis=None)

**Observations**
These observations are about the 30 grid candidates (5 feature sets x 3 models x 2 targets). Sections 6 and 7 add 4 more.
- **Log price is the largest single gain in MdAPE.** It decreases MdAPE for every XGBoost candidate by 0.3 to 0.5 points, and for every random forest by only 0.1 to 0.2 points. The 10 XGBoost log-price candidates are the 10 best grid candidates.
- **Comparable sales help with HOA and flooring in the set.** For XGBoost (06 settings, log price), they decrease MdAPE from 7.35% to 7.21%.
- **HOA and flooring help with or without comparable sales.** With MdAPE as the metric, they are better than the reference for the random forest in all 6 months, also without comps (7.67% against 7.85%, dollar price). With R² as the metric, they made the random forest worse.
- **The 06 settings are better than the team settings** for 8 of the 10 feature set and target pairs. In the other 2 (MLS district + HOA + flooring), the difference is 0.01 points or less.
- **School districts against the MLS district:** with HOA, flooring and comps on log price, the two are equal (7.21% and 7.20%, lower in 3 of 6 months). We keep the school districts, because they do not depend on what the agent types in the listing.
- **Overfitting is moderate.** The best grid candidate has a training MdAPE of 5.9% and a validation MdAPE of 7.2%. The random forest has a larger difference (4.2% against 7.5% for its best version).


## 3. The leader against the next candidates
This section compares the months one by one, so that a gain in one month cannot carry the mean. Each row compares a candidate with the leader. A positive gain means that the leader had the lower MdAPE. "Months improved" counts the months in which the candidate was better than the leader.

In [6]:
leader = summary.index[0]
rivals = gain_summary(scores, reference=leader, metric=DECISION_METRIC).sort_values("mean gain")
print("leader:", leader)
rivals.head(6)

leader: XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price


,mean gain,std gain,months improved (of 6)
model,,,
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,0.0006,0.0005,1
XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,0.0010,0.0005,0
XGBoost (06 settings) | Districts + HOA + flooring + similar comps | log price,0.0013,0.0005,0
XGBoost (06 settings) | MLS district + HOA + flooring + comps | log price,0.0019,0.0012,0
XGBoost (06 settings) | Districts + HOA + flooring + comps | log price,0.0019,0.0012,0
XGBoost (team settings) | Districts + HOA + flooring + comps | log price,0.0030,0.0007,0


In [7]:
by_month = scores.pivot(index="month", columns="model", values=DECISION_METRIC)
by_month[[leader, *rivals.index[:3], REFERENCE]].style.format("{:.2%}")

model,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,XGBoost (06 settings) | Districts + HOA + flooring + similar comps | log price,Random Forest | Districts + comps | dollar price
month,,,,,
2025-10,6.99%,7.02%,7.15%,7.14%,7.83%
2025-11,6.93%,6.93%,7.09%,7.10%,7.81%
2025-12,7.09%,7.15%,7.17%,7.26%,7.79%
2026-01,7.06%,7.20%,7.09%,7.09%,8.07%
2026-02,7.09%,7.17%,7.18%,7.24%,7.96%
2026-03,6.92%,6.96%,7.01%,7.07%,7.68%


**Observations**
- **Leader (all 34 candidates):** XGBoost (06 settings) + school districts + HOA + flooring + comparable sales + their median sale price, log price. MdAPE 7.02%, R² 0.9144, 63.6% of homes within 10%. It is candidate C of section 6.
- **The next candidates** are C + room ratios (section 7, +0.06 points, lower than the leader in 1 of 6 months) and the plain + similar comps candidate (+0.10 points, lower in 0 of 6 months).
- **The served model** (the best grid candidate, MdAPE 7.21%) has a higher MdAPE than the leader in all 6 months.
- **Against the reference** (random forest + districts + comps, dollar price, 7.85%): 0.84 points lower, in all 6 months.


## 4. Ensembles
An ensemble here is the mean of the log predictions of some candidates, with equal weights (a geometric mean). The members are fit on log price, so their explanations (SHAP values) also average on this scale. Nothing is fit, so the ensemble cannot learn the validation months. The members are selected by two rules that we fixed before we saw the results:

- **Best XGBoost + best random forest:** two different model types, so their errors are less similar.
- **Top 3 candidates:** the three candidates with the lowest MdAPE.

The members are selected on the same validation months that score the ensemble. This gives a small optimistic bias. The test month stays the final check.

In [8]:
def best_of(estimator):
    return next(name for name in summary.index if name.startswith(estimator))


ENSEMBLES = {
    "Ensemble | best XGBoost + best random forest": [best_of("XGBoost"), best_of("Random Forest")],
    "Ensemble | top 3 candidates": list(summary.index[:3]),
}
eval_month = df.loc[predictions.index, DATE_COL].dt.to_period("M").astype(str)
rows = []
for name, members in ENSEMBLES.items():
    blend = np.exp(np.log(predictions[members]).mean(axis=1))  # mean on the log scale, like the members' target
    rows += [{"model": name, "month": m, **score(actual[idx], blend[idx])} for m, idx in eval_month.groupby(eval_month).groups.items()]
    print(f"{name}: {' + '.join(members)}")
ensemble_scores = pd.DataFrame(rows)
with_leader = pd.concat([scores[scores["model"] == leader], ensemble_scores], ignore_index=True)
with_leader.groupby("model")[METRICS].mean().join(gain_summary(with_leader, reference=leader, metric=DECISION_METRIC))

Ensemble | best XGBoost + best random forest: XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price + Random Forest | Districts + HOA + flooring | log price
Ensemble | top 3 candidates: XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price + XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price + XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price


,median_ape,r2,mae,mape,within_10pct,mean gain,std gain,months improved (of 6)
model,,,,,,,,
Ensemble | best XGBoost + best random forest,0.0702,0.9095,"135,545.1832",0.1048,0.6370,0.0001,0.0007,3.0000
Ensemble | top 3 candidates,0.0688,0.9176,"131,469.1378",0.1026,0.6437,-0.0013,0.0003,6.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,0.0702,0.9144,"133,820.4476",0.1044,0.6358,NaN,NaN,NaN


Correlation of the log errors of the members. An ensemble helps more when its members make different errors (low correlation).

In [9]:
members = sorted({m for ms in ENSEMBLES.values() for m in ms})
np.log(predictions[members].div(actual, axis=0)).corr().round(3)

,Random Forest | Districts + HOA + flooring | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price
Random Forest | Districts + HOA + flooring | log price,1.0000,0.8670,0.8650,0.8780
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,0.8670,1.0000,0.9670,0.9330
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,0.8650,0.9670,1.0000,0.9350
XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,0.8780,0.9330,0.9350,1.0000


**Observations**
- **Top 3** (C, C + ratios, and plain + similar comps): MdAPE 6.88%, 0.14 points lower than the leader, lower in all 6 months. R² 0.9176, MAE \$131.5K, 64.4% within 10%. It passes the decision rule.
- **The members are fixed by the rule** (the three best candidates), so C + ratios is a member although it does not pass section 7 on its own. The top 3 of the earlier run (C, B, A) gives almost the same result (6.90%).
- **Its members are different enough to help:** the correlation of log errors is 0.93 to 0.97. The pairs that use different comparable sales have the lowest correlation.
- **Best XGBoost + best random forest:** MdAPE 7.02%, a tie with the leader, and R² is lower. It does not pass.
- **Bias:** the members were selected on the same validation months. The test month was already used for the earlier selection, so it cannot give a fresh check of this ensemble.


## 5. Price bands
Is a different candidate best for cheap and for expensive homes? The sale price is not known at prediction time, so the bands use the leader's estimate. Bands are fifths (quintiles) of that estimate.

In [10]:
estimate = predictions[leader]
pct_error = predictions.div(actual, axis=0).sub(1).abs()
band = pd.qcut(estimate, 5, labels=["Lowest fifth", "Second", "Middle", "Fourth", "Highest fifth"])
shown = list(summary.index[:4]) + [best_of("Random Forest")]
by_band = pct_error[shown].groupby(band, observed=True).median()
by_band.insert(0, "estimate up to", estimate.groupby(band, observed=True).max().map("${:,.0f}".format))
by_band.style.format("{:.2%}", subset=shown).highlight_min(subset=shown, axis=1, color="#dbe4ff")

,estimate up to,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,XGBoost (06 settings) | Districts + HOA + flooring + similar comps | log price,Random Forest | Districts + HOA + flooring | log price
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,,,,,,
Lowest fifth,"$575,390",6.25%,6.31%,6.23%,6.32%,6.42%
Second,"$782,738",5.43%,5.42%,5.55%,5.55%,5.64%
Middle,"$1,040,721",6.34%,6.37%,6.37%,6.39%,6.80%
Fourth,"$1,553,663",8.16%,8.15%,8.30%,8.32%,8.77%
Highest fifth,"$8,741,204",10.16%,10.30%,10.26%,10.40%,11.29%


**A fair test of one model for each band.** If we select the best candidate for each band on the same months that score it, the result is too optimistic. Thus:
1. Select the best candidate for each band on the first 3 validation months (2025-10 to 2025-12). The band limits also come from these months.
2. Score the combined ("routed") model on the last 3 months (2026-01 to 2026-03), which had no part in the selection.
3. Compare with the leader on the same 3 months.

In [11]:
month = df.loc[predictions.index, DATE_COL].dt.to_period("M")
first = (month <= EVAL_MONTHS[2]).to_numpy()
limits = estimate[first].quantile([0.2, 0.4, 0.6, 0.8]).to_numpy()
band_of = pd.Series(np.searchsorted(limits, estimate), index=estimate.index)
winners = pct_error[first].groupby(band_of[first]).median().idxmin(axis=1)
routed = pd.Series(np.nan, index=estimate.index)
for b, name in winners.items():
    routed[band_of == b] = predictions.loc[band_of == b, name]
print("winner for each band (selected on 2025-10 to 2025-12):")
for b, name in winners.items():
    print(f"  band {b + 1}: {name}")

later = ~first
rows = []
for name, values in {leader: predictions[leader], "Routed by price band": routed}.items():
    for m, idx in eval_month[later].groupby(eval_month[later]).groups.items():
        rows.append({"model": name, "month": m, **score(actual[idx], values[idx])})
routed_scores = pd.DataFrame(rows)
routed_scores.groupby("model")[METRICS].mean().join(gain_summary(routed_scores, reference=leader, metric=DECISION_METRIC))

winner for each band (selected on 2025-10 to 2025-12):
  band 1: XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price
  band 2: XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price
  band 3: XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price
  band 4: XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price
  band 5: XGBoost (team settings) | Districts + HOA + flooring + comps | log price


,median_ape,r2,mae,mape,within_10pct,mean gain,std gain,months improved (of 3)
model,,,,,,,,
Routed by price band,0.0711,0.9149,"136,358.2903",0.1049,0.6315,0.0008,0.0004,0.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,0.0703,0.9170,"134,878.1894",0.1043,0.6344,NaN,NaN,NaN


**Observations**
- **The error increases with the price.** For the leader, the median error is 5.4% to 6.3% in the three lowest fifths, 8.2% in the fourth fifth and 10.2% in the highest fifth (estimates above approximately \$1.55M).
- **A different model for each band does not help.** The winners (selected on 2025-10 to 2025-12) are the leader for 3 bands, C + ratios for 1 band and XGBoost (team settings) for the highest band. On 2026-01 to 2026-03, the routed model has an MdAPE of 7.11%, against 7.03% for the leader, and it is worse in all 3 months. We do not use routing by price band.


## 6. Similar comparable sales (test fixed in advance)
The plain comparable sales are the 10 **nearest** earlier sales. The similar comparable sales are the 10 most **similar** homes among the 50 nearest earlier sales. Similarity is the sum of the differences in fixed units from appraisal rules of thumb: 25% of living area, 20 years of age, 1 bedroom, 1 bathroom, a factor of 2 in lot size and 1 km of distance (`features.SIMILARITY_UNITS`). We fixed these units before we saw any result, so they are not fit to the validation months.

Three candidates, each XGBoost (06 settings) on log price with districts, HOA and flooring:
- **A:** similar comparables instead of the plain comparables.
- **B:** both.
- **C:** the plain comparables plus their median sale price (no size adjustment).

**Decision rule (fixed in advance):** a candidate replaces the selected model if its mean MdAPE is lower and its MdAPE is lower in at least 5 of the 6 months.

In [12]:
SELECTED = "XGBoost (06 settings) | Districts + HOA + flooring + comps | log price"
NEW = [name for name in CANDIDATES if CANDIDATES[name][1] in SIMILAR_SETS]
test = scores[scores["model"].isin([SELECTED, *NEW])]
test.groupby("model")[METRICS + TRAIN_METRICS].mean().join(gain_summary(test, reference=SELECTED, metric=DECISION_METRIC))

,median_ape,r2,mae,mape,within_10pct,train_median_ape,train_r2,mean gain,std gain,months improved (of 6)
model,,,,,,,,,,
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,0.0702,0.9144,"133,820.4476",0.1044,0.6358,0.0573,0.9620,-0.0019,0.0012,6.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,0.0711,0.9142,"135,091.0967",0.1055,0.6317,0.0571,0.9625,-0.0010,0.0009,5.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps | log price,0.0721,0.9133,"135,999.7839",0.1063,0.6278,0.0587,0.9600,NaN,NaN,NaN
XGBoost (06 settings) | Districts + HOA + flooring + similar comps | log price,0.0715,0.9109,"136,353.2836",0.1061,0.6316,0.0581,0.9601,-0.0006,0.0009,4.0000


In [13]:
test.pivot(index="month", columns="model", values=DECISION_METRIC)[[SELECTED, *NEW]].style.format("{:.2%}")

model,XGBoost (06 settings) | Districts + HOA + flooring + comps | log price,XGBoost (06 settings) | Districts + HOA + flooring + similar comps | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price
month,,,,
2025-10,7.15%,7.14%,7.15%,6.99%
2025-11,7.28%,7.10%,7.09%,6.93%
2025-12,7.22%,7.26%,7.17%,7.09%
2026-01,7.09%,7.09%,7.09%,7.06%
2026-02,7.39%,7.24%,7.18%,7.09%
2026-03,7.12%,7.07%,7.01%,6.92%


**Observations**
- **Two of the three candidates pass the decision rule** against the selected model (MdAPE 7.21%):
  - **C (plain comps + median comparable price): MdAPE 7.02%, -0.19 points, lower in 6 of 6 months**, R² 0.9144.
  - B (plain + similar comps): 7.11%, -0.10 points, lower in 5 of 6 months, R² 0.9142.
  - A (similar comps only): 7.15%, -0.06 points, lower in only 4 of 6 months, R² 0.9109. It does not pass.
- **C is the best of the three.** The median comparable price has no size adjustment. It gives the model the price level of the neighborhood directly, which helps when the living area of the home or of its neighbors is wrong or unusual.
- **The similar comparables are a better estimate on their own** (median error 9.7% against 13.2% for the plain comparables, before any model), but in the model the gain is smaller, because the model already adjusts for the home's own attributes.
- **Run on the corrected Dataset A (2026-11-22).** In the first run, all three passed (A in 6 of 6 months). After the duplicate-listing fix and the team value rules, A passes in only 4 months. C and B keep their result.
- **The test month is already used.** These candidates were added after the test month was scored. Thus, the test month cannot give an independent check of them; validation is the basis for this decision.


## 7. Room ratios (team default group)
The team's unified pipeline includes room ratios in its default feature groups: bedrooms per bathroom, square feet per bedroom and the log of lot size over living area (`features.RATIO_COLS`, the team's definitions). In week 6, the ratios made the random forest worse in all months, but that test used R². This section tests them with the decision metric, on the best candidate of section 6.

- **D:** C + room ratios.

**Decision rule (fixed in advance):** D replaces C if its mean MdAPE is lower and its MdAPE is lower in at least 5 of the 6 months.

In [14]:
C = "XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price"
D = "XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price"
ratio_test = scores[scores["model"].isin([C, D])]
ratio_test.groupby("model")[METRICS + TRAIN_METRICS].mean().join(gain_summary(ratio_test, reference=C, metric=DECISION_METRIC))

,median_ape,r2,mae,mape,within_10pct,train_median_ape,train_r2,mean gain,std gain,months improved (of 6)
model,,,,,,,,,,
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,0.0707,0.9145,"134,086.2215",0.1047,0.6340,0.0568,0.9633,0.0006,0.0005,1.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,0.0702,0.9144,"133,820.4476",0.1044,0.6358,0.0573,0.9620,NaN,NaN,NaN


In [15]:
ratio_test.pivot(index="month", columns="model", values=DECISION_METRIC)[[C, D]].style.format("{:.2%}")

model,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price
month,,
2025-10,6.99%,7.02%
2025-11,6.93%,6.93%
2025-12,7.09%,7.15%
2026-01,7.06%,7.20%
2026-02,7.09%,7.17%
2026-03,6.92%,6.96%


**Observations**
- **D does not pass.** MdAPE 7.07% against 7.02% for C, lower in only 1 of 6 months. R² is the same (0.9145 against 0.9144).
- **This agrees with week 6**, where the ratios made the random forest worse. The ratios are combinations of columns that the trees already have (bedrooms, bathrooms, living area, lot size), so they add no new information.
- We do not add the room ratios to the selected model.


## Summary
- **Data:** Dataset A after the 2026-11-22 fix (one row for each `ListingKey`, and the team's value rules). All 34 candidates were fit again on it.
- **Best single model (34 candidates):** XGBoost (06 settings) on Dataset A + school districts + HOA + flooring + comparable sales + their median sale price, log price. Validation MdAPE 7.02%, R² 0.9144, 63.6% within 10%.
- **Against the model that is served now** (the same without the median comparable price, MdAPE 7.21%): 0.19 points lower, in all 6 months.
- **Ensemble of the top 3** (C, C + ratios, plain + similar comps; log-scale mean): MdAPE 6.88%, R² 0.9176, lower MdAPE than C in all 6 months.
- **Room ratios (team default group):** no gain. **Routing by price band:** no gain.
- **Caution:** the test month (2026-04) was scored for the served model before these candidates existed. A new test score for any of them is a second look at the test month.


In [16]:
all_scores = pd.concat([scores, ensemble_scores], ignore_index=True)
final = all_scores.groupby("model")[METRICS + TRAIN_METRICS].mean()
final = final.join(gain_summary(all_scores, reference=REFERENCE, metric=DECISION_METRIC)).sort_values(DECISION_METRIC)
final.reset_index().to_csv(RESULTS_DIR / "final_model_selection.csv", index=False)
final.head(8)

,median_ape,r2,mae,mape,within_10pct,train_median_ape,train_r2,mean gain,std gain,months improved (of 6)
model,,,,,,,,,,
Ensemble | top 3 candidates,0.0688,0.9176,"131,469.1378",0.1026,0.6437,NaN,NaN,-0.0097,0.0012,6.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price | log price,0.0702,0.9144,"133,820.4476",0.1044,0.6358,0.0573,0.9620,-0.0084,0.0011,6.0000
Ensemble | best XGBoost + best random forest,0.0702,0.9095,"135,545.1832",0.1048,0.6370,NaN,NaN,-0.0083,0.0012,6.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps + comp median price + ratios | log price,0.0707,0.9145,"134,086.2215",0.1047,0.6340,0.0568,0.9633,-0.0078,0.0009,6.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps + similar comps | log price,0.0711,0.9142,"135,091.0967",0.1055,0.6317,0.0571,0.9625,-0.0074,0.0013,6.0000
XGBoost (06 settings) | Districts + HOA + flooring + similar comps | log price,0.0715,0.9109,"136,353.2836",0.1061,0.6316,0.0581,0.9601,-0.0070,0.0015,6.0000
XGBoost (06 settings) | MLS district + HOA + flooring + comps | log price,0.0720,0.9123,"136,370.3106",0.1062,0.6291,0.0586,0.9598,-0.0065,0.0018,6.0000
XGBoost (06 settings) | Districts + HOA + flooring + comps | log price,0.0721,0.9133,"135,999.7839",0.1063,0.6278,0.0587,0.9600,-0.0064,0.0017,6.0000
